<a href="https://colab.research.google.com/github/nikidhacse/DR-PET/blob/main/Research_Paper_Answer_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Research Paper Answer Bot

### A Retrieval-Augmented Generation (RAG) System for Research Papers

**Capstone Project — GenAI Pinnacle Plus Program**

---

## Project Objective

This project aims to build an intelligent question-answering system over a collection of research papers.

The system uses Retrieval-Augmented Generation (RAG) to:

- Load and process research papers.
- Split documents into meaningful text chunks.
- Convert text into embeddings.
- Store embeddings in a vector database.
- Retrieve relevant research-paper passages for a user query.
- Use a Large Language Model (LLM) to generate a grounded answer.
- Display the top-3 supporting sources, including the paper title and page number.

The project also compares different embedding models and retrieval strategies to identify the approach that provides the best retrieval quality.

In [4]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', 'patient real time data.gsheet', 'hospital records .gsheet', 'Untitled spreadsheet.gsheet', 'Problem crushers ', 'BCM IAE-1 Answer (1).gdoc', 'BCM IAE-1 Answer.gdoc', 'Course .gdoc', 'Unt.gdoc', 'linkedin post .gdoc', 'linkedin profile.gdoc', 'prompting with GPT.gdoc', 'CYBER THREAT LANDSCAPE ANALYSIS REPORT .gdoc', 'BMC ', 'LINKEDIN POST (3).gdoc', '10 hr course IBM.gdoc', 'Untitled document (4).gdoc', 'Revolutionising-E-commerce-Visualisation-with-Multimodal-AI (1) - Copy.pdf', 'PPT ', 'LINKEDIN CONNECTION  (1).gdoc', 'Part A qp no.1.gdoc', 'Untitled document (3).gdoc', '10 hrs course.gdoc', 'linkedin connection .gdoc', '10HRS COURSE.gdoc', 'LINKEDIN POST (2).gdoc', 'Copy of Engineering Chemistry QB with Answer.pdf', 'LINKEDIN CONNECTION .gdoc', 'LINKEDIN POST (1).gdoc', '10 HOURS COURSE:.gdoc', 'Flask-Based Stock Market Analysis API Project.gdoc', 'LINKEDLN CONNECTION  .gdoc', 'LINKEDIN POST.gdoc', 'Analytics vidhya.gdoc', 'AWS CERTIFICATE .gdoc', 'DTI BOOK.gdoc'

In [6]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', 'patient real time data.gsheet', 'hospital records .gsheet', 'Untitled spreadsheet.gsheet', 'Problem crushers ', 'BCM IAE-1 Answer (1).gdoc', 'BCM IAE-1 Answer.gdoc', 'Course .gdoc', 'Unt.gdoc', 'linkedin post .gdoc', 'linkedin profile.gdoc', 'prompting with GPT.gdoc', 'CYBER THREAT LANDSCAPE ANALYSIS REPORT .gdoc', 'BMC ', 'LINKEDIN POST (3).gdoc', '10 hr course IBM.gdoc', 'Untitled document (4).gdoc', 'Revolutionising-E-commerce-Visualisation-with-Multimodal-AI (1) - Copy.pdf', 'PPT ', 'LINKEDIN CONNECTION  (1).gdoc', 'Part A qp no.1.gdoc', 'Untitled document (3).gdoc', '10 hrs course.gdoc', 'linkedin connection .gdoc', '10HRS COURSE.gdoc', 'LINKEDIN POST (2).gdoc', 'Copy of Engineering Chemistry QB with Answer.pdf', 'LINKEDIN CONNECTION .gdoc', 'LINKEDIN POST (1).gdoc', '10 HOURS COURSE:.gdoc', 'Flask-Based Stock Market Analysis API Project.gdoc', 'LINKEDLN CONNECTION  .gdoc', 'LINKEDIN POST.gdoc', 'Analytics vidhya.gdoc', 'AWS CERTIFICATE .gdoc', 'DTI BOOK.gdoc'

In [7]:
import os

research_paper_path = "/content/drive/MyDrive/research_papers"

print(os.listdir(research_paper_path))

['01_Attention_Is_All_You_Need.pdf', '01_Chain_of_Thought.pdf', '03_Tree_of_Thoughts.pdf', '02_Least_to_Most.pdf', '03_RoBERTa.pdf', '02_Dense_Passage_Retrieval.pdf', '03_ANCE.pdf', '02_BERT.pdf', '03_E5_Embeddings.pdf', '01_Sentence_BERT.pdf', '03_PaLM.pdf', '01_GPT3.pdf', '01_FAISS.pdf', '02_BGE_M3.pdf', '02_LLaMA.pdf', '03_REALM.pdf', '02_ColBERT.pdf', '02_Retrieval_Augmented_Generation.pdf', 'chroma_db']


In [2]:
# Install the main libraries required for the RAG project

!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-openai \
    chromadb \
    pypdf \
    sentence-transformers \
    rank-bm25 \
    faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.7/123.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.9/565.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/13

In [3]:
# Import the libraries we will use in the RAG project

import os
import sys

import pypdf
import chromadb
import faiss

import langchain
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

print("✅ Python version:", sys.version.split()[0])
print("✅ LangChain version:", langchain.__version__)
print("✅ PyPDF imported successfully")
print("✅ ChromaDB imported successfully")
print("✅ FAISS imported successfully")
print("✅ Sentence Transformers imported successfully")
print("\n🎉 RAG environment is ready!")

/tmp/ipykernel_844/1517542762.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ Python version: 3.12.13
✅ LangChain version: 1.3.13
✅ PyPDF imported successfully
✅ ChromaDB imported successfully
✅ FAISS imported successfully
✅ Sentence Transformers imported successfully

🎉 RAG environment is ready!


In [8]:
from langchain_community.document_loaders import PyPDFLoader
import os

research_paper_path = "/content/drive/MyDrive/research_papers"

all_documents = []

for filename in os.listdir(research_paper_path):
    if filename.lower().endswith(".pdf"):
        file_path = os.path.join(research_paper_path, filename)

        print(f"📄 Loading: {filename}")

        loader = PyPDFLoader(file_path)
        documents = loader.load()

        all_documents.extend(documents)

print("\n" + "="*50)
print("✅ ALL RESEARCH PAPERS LOADED!")
print("📚 Number of PDF files:", len([
    f for f in os.listdir(research_paper_path)
    if f.lower().endswith(".pdf")
]))
print("📄 Total pages loaded:", len(all_documents))
print("="*50)

📄 Loading: 01_Attention_Is_All_You_Need.pdf
📄 Loading: 01_Chain_of_Thought.pdf
📄 Loading: 03_Tree_of_Thoughts.pdf
📄 Loading: 02_Least_to_Most.pdf
📄 Loading: 03_RoBERTa.pdf
📄 Loading: 02_Dense_Passage_Retrieval.pdf
📄 Loading: 03_ANCE.pdf


📄 Loading: 02_BERT.pdf
📄 Loading: 03_E5_Embeddings.pdf
📄 Loading: 01_Sentence_BERT.pdf
📄 Loading: 03_PaLM.pdf
📄 Loading: 01_GPT3.pdf
📄 Loading: 01_FAISS.pdf
📄 Loading: 02_BGE_M3.pdf
📄 Loading: 02_LLaMA.pdf
📄 Loading: 03_REALM.pdf
📄 Loading: 02_ColBERT.pdf
📄 Loading: 02_Retrieval_Augmented_Generation.pdf

✅ ALL RESEARCH PAPERS LOADED!
📚 Number of PDF files: 18
📄 Total pages loaded: 472


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(all_documents)

print("✅ TEXT CHUNKING COMPLETE!")
print("📄 Total pages:", len(all_documents))
print("🧩 Total chunks:", len(chunks))
print("=" * 50)

✅ TEXT CHUNKING COMPLETE!
📄 Total pages: 472
🧩 Total chunks: 2213


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded successfully!


In [11]:
from langchain_community.vectorstores import Chroma

# Create the vector database from our document chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="/content/drive/MyDrive/research_papers/chroma_db"
)

print("✅ Vector database created successfully!")
print("📚 Number of chunks stored:", len(chunks))

✅ Vector database created successfully!
📚 Number of chunks stored: 2213


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Load the SECOND embedding model
embedding_model_2 = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

print("✅ Second embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Second embedding model loaded successfully!


In [1]:
vectorstore_2 = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model_2,
    persist_directory="/content/drive/MyDrive/research_papers/chroma_db_2"
)

print("✅ Second vector database created successfully!")
print("📚 Number of chunks stored:", len(chunks))

NameError: name 'Chroma' is not defined